# abla-qwen.ipynb

Qwen2-Audio speech-only, text-only, and multimodal ablation experiments.

Set the input and per-modality output paths, then run the selected modality section and its comparisons.

Use the setup steps in the repository README before running this notebook.


In [ ]:
# ============================================================
# QWEN2-AUDIO BASELINE DEVIATION RUN
# FULL VERSION WITH TASK-AWARE PROMPTS + SELECTED LAYER SAVING
# ============================================================

import os
import re
import json
import random
import librosa
import torch
import pandas as pd
from tqdm import tqdm
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor

# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "Qwen/Qwen2-Audio-7B-Instruct"
BASE_DIR = "/content/drive/MyDrive/probing"
#SAVE_DIR = "/content/drive/MyDrive/probing/baseline_outputs_edited"

SAVE_DIR = "/content/drive/MyDrive/probing/full_probe/albe"
os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SELECTED_AUDIO_LAYERS = [4, 16, 31]
SELECTED_LM_LAYERS = [8, 20, -1]

# ============================================================
# PROMPTS
# ============================================================

# Delivery
SYSTEM_PROMPT_DELIVERY = """
You are a speech emotion classifier.

Determine whether the speaker sounds neutral or emotional.

Choices:
[neutral, emotional]

Return ONLY valid JSON:
{"label": "<neutral or emotional>", "confidence": <0.0-1.0>}
"""

USER_PROMPT_DELIVERY = "Classify the speech."

# Intensity
def build_intensity_prompt(target_emotion, related_emotion=None):

    if related_emotion:
        choices = f"[neutral, {target_emotion}, {related_emotion}]"
    else:
        choices = f"[neutral, {target_emotion}]"

    return f"""
You are a speech emotion classifier.

Determine the emotion of the speaker.

Choices:
{choices}

Return ONLY valid JSON:
{{"label": "<...>", "confidence": <0.0-1.0>}}
"""

USER_PROMPT_INTENSITY = "Classify the emotion of the speech."



# ============================================================
# MODEL LOAD
# ============================================================

processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype="float16",
    device_map="auto"
)

model.eval()

In [ ]:
for name, module in model.model.named_children():
    print(name)

In [ ]:
model.model.named_children

In [ ]:
import gc
import torch

def clear_memory():
    """Clear PyTorch cache and force garbage collection"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# ============================================================
# PATH REMAP
# ============================================================

def remap_to_colab_path(old_path):

    if not isinstance(old_path, str):
        raise ValueError(f"Invalid path: {old_path}")

    # normalize Windows → Unix
    old_path = old_path.replace("\\", "/").strip()

    # join with Drive base
    new_path = os.path.join(BASE_DIR, old_path)

    return new_path

# ============================================================
# JSON SAFE PARSER
# ============================================================


def parse_json_safe(text):

    if not isinstance(text, str):
        return None, None

    raw = text.strip()

    # ---- remove markdown fences ----
    raw = re.sub(r"^```json", "", raw, flags=re.IGNORECASE).strip()
    raw = re.sub(r"^```", "", raw).strip()
    raw = re.sub(r"```$", "", raw).strip()

    # ---- try strict JSON first ----
    try:
        return _parse_valid_json(raw)
    except:
        pass

    # ---- extract JSON substring ----
    start = raw.find("{")
    end = raw.rfind("}")

    if start != -1 and end != -1:
        json_str = raw[start:end+1]

        # fix common issues
        json_str = re.sub(r"(\w+):", r'"\1":', json_str)   # label: → "label":
        json_str = json_str.replace("'", '"')              # ' → "
        json_str = re.sub(r",\s*}", "}", json_str)

        try:
            return _parse_valid_json(json_str)
        except:
            pass

    # ---- fallback: regex extraction ----
    label = None
    confidence = None

    # label
    label_match = re.search(
        r"(neutral|emotional|happy|sad|angry|fearful|disgust|caring|supportive|curious)",
        raw.lower()
    )

    if label_match:
        label = label_match.group(1)

    # confidence
    conf_match = re.search(r"(0\.\d+|1\.0+)", raw)

    if conf_match:
        confidence = float(conf_match.group(1))

    # ---- debug print ----
    #print("⚠️ Fallback parsing used")
    #print("RAW OUTPUT:", raw)

    return label, confidence



def _parse_valid_json(text):
    label, confidence = json.loads(text)
    return label, confidence

# ============================================================
# AUDIO LOAD (UPDATED FOR AUDIO FLAMINGO)
# ============================================================

def load_audio(wav_path):
    # Audio Flamingo expects raw numpy array
    audio, sr = librosa.load(wav_path, sr=16000)  # AF3 uses 16kHz
    return audio

# ============================================================
# HOOK STORAGE
# ============================================================

activations = {}

def make_hook(name):
    def hook(module, inp, out):
        if isinstance(out, tuple):
            out = out[0]
        activations[name] = out.detach().cpu()
    return hook

hooks = []

# def select_three_regions(num_layers):
#     """
#     Return 3 early, 3 mid, 3 late layer indices
#     """
#     if num_layers < 9:
#         raise ValueError(f"Too few layers ({num_layers}) to select 3x3 regions")

#     early = list(range(0, 3))

#     mid_start = num_layers // 2 - 1
#     mid = list(range(mid_start, mid_start + 3))

#     late = list(range(num_layers - 3, num_layers))

#     return early + mid + late
def select_three_regions(num_layers):
    """
    Return ALL layers (modified from original)
    """
    return list(range(num_layers))
num_lm_layers = len(model.model.language_model.layers)
print("number of layers:",num_lm_layers)
SELECTED_LM_LAYERS = select_three_regions(num_lm_layers)
# LM hooks (AF3 uses Qwen2 with 28 layers)
for idx in SELECTED_LM_LAYERS:
    h = model.model.language_model.layers[idx].register_forward_hook(
        make_hook(f"lm_{idx}")
    )
    hooks.append(h)

# ============================================================
# POOLING
# ============================================================

def mean_pool(x):
    if x.ndim == 3:
        return x.mean(dim=1).squeeze(0)
    elif x.ndim == 2:
        return x.mean(dim=0)
    else:
        return x.flatten()



# ============================================================
# MODEL INFERENCE
# ============================================================

# ============================================================
# HOOK STORAGE
# ============================================================

import gc
import torch

activations = {}

def make_hook(name):
    def hook(module, inp, out):

        if isinstance(out, tuple):
            out = out[0]

        # out : (B, T, H)

        with torch.no_grad():

            pooled = out.mean(dim=1).detach().cpu()

        activations[name] = pooled

    return hook


hooks = []

# ============================================================
# SELECT LAYERS
# ============================================================

def select_three_regions(num_layers):

    # use every layer
    return list(range(num_layers))

num_lm_layers = len(model.model.language_model.layers)

print("Number of LM layers:", num_lm_layers)

SELECTED_LM_LAYERS = select_three_regions(num_lm_layers)

for idx in SELECTED_LM_LAYERS:

    hooks.append(

        model.model.language_model.layers[idx].register_forward_hook(

            make_hook(f"lm_{idx}")

        )

    )

# ============================================================
# REPRESENTATION EXTRACTION (NO HOOKS)
# ============================================================

import gc
import librosa
import torch


def select_layers(num_layers):
    """
    Return all LM layers.
    Change this if you later want only early/mid/late.
    """
    return list(range(num_layers))


def judge_qwen2_audio(
    wav_path,
    system_prompt,
    user_prompt,
):

    # --------------------------------------------------------
    # LOAD AUDIO
    # --------------------------------------------------------

    waveform, _ = librosa.load(
        wav_path,
        sr=processor.feature_extractor.sampling_rate
    )

    # --------------------------------------------------------
    # CONVERSATION
    # --------------------------------------------------------

    conversation = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "audio",
                    "audio_url": f"file://{wav_path}"
                },
                {
                    "type": "text",
                    "text": user_prompt
                }
            ]
        }
    ]

    text = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=False
    )

    # --------------------------------------------------------
    # TOKENIZE
    # --------------------------------------------------------

    inputs = processor(
        text=text,
        audio=[waveform],
        sampling_rate=processor.feature_extractor.sampling_rate,
        return_tensors="pt",
        padding=True
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    # --------------------------------------------------------
    # FORWARD
    # --------------------------------------------------------

    with torch.no_grad():

        outputs = model(

            **inputs,

            output_hidden_states=True,

            return_dict=True,

            use_cache=False
        )

    # --------------------------------------------------------
    # EXTRACT REPRESENTATIONS
    # --------------------------------------------------------

    pooled_reps = {}

    hidden_states = outputs.hidden_states

    selected_layers = select_layers(len(hidden_states))

    for i in selected_layers:

        pooled = (
            hidden_states[i]
            .mean(dim=1)
            .detach()
            .cpu()
        )

        pooled_reps[f"lm_{i}"] = pooled

    # --------------------------------------------------------
    # CLEANUP
    # --------------------------------------------------------

    del hidden_states
    del outputs
    del inputs
    del waveform

    gc.collect()
    torch.cuda.empty_cache()

    return pooled_reps
# ============================================================
# SAVE REPRESENTATIONS
# ============================================================

def save_representation_tensor(rep_dict, save_path):
    torch.save(rep_dict, save_path)

# ============================================================
# SAMPLE SUBSET PER TASK
# ============================================================

def sample_subset(paths, fraction):
    n = max(10, int(len(paths) * fraction))
    return random.sample(paths, min(n, len(paths)))



In [ ]:
import os
import json
import torch
import pandas as pd
from tqdm import tqdm

BASE_DIR = "/content/drive/MyDrive/subprobe"

def resolve_path(p):
    p = str(p).replace("\\", "/").strip()

    if p.startswith("/content/"):
        return p

    return os.path.join(BASE_DIR, p)


CSV_PATH = os.path.join(BASE_DIR, "merged_subtle_emotion.csv")

SAVE_DIR = os.path.join(
    BASE_DIR,
    "task_ablation_qwen/speech"
)
os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(CSV_PATH)

# random subset
df = df.sample(n=1000, random_state=42)

print("Total samples:", len(df))

# ============================================================
# RESUME
# ============================================================

processed = {
    os.path.splitext(f)[0]
    for f in os.listdir(SAVE_DIR)
    if f.endswith(".pt")
}

print(f"Already processed: {len(processed)}")

# ============================================================
# PROMPTS
# ============================================================

SYSTEM_PROMPT = "You are a helpful assistant."

BASE_PROMPT = (
    "What is the emotion in this speech?"
)

# ============================================================
# RUN
# ============================================================

rows = []
csv_out = os.path.join(
    SAVE_DIR,
    "task3_results.csv"
)

write_header = not os.path.exists(csv_out)
for idx, row in tqdm(df.iterrows(), total=len(df)):

    # ----------------------------------------
    # Skip processed samples
    # ----------------------------------------

    if str(idx) in processed:
        continue

    wav_path = resolve_path(row["audio_path"])

    if not os.path.exists(wav_path):
        print(f"Missing audio: {wav_path}")
        continue

    # ----------------------------------------
    # Parse labels
    # ----------------------------------------

    labels = json.loads(row["labels"])
    counts = json.loads(row["label_counts"])

    transcript = str(row["Utterance"]).strip()

    #current_prompt = (
    #    BASE_PROMPT +
    #    "\n\nTranscript: " +
    #    transcript
    #)
    current_prompt=BASE_PROMPT


    # ----------------------------------------
    # Inference
    # ----------------------------------------

    try:

        reps = judge_qwen2_audio(
            wav_path,
            SYSTEM_PROMPT,
            current_prompt
        )

        # save representation

        rep_path = os.path.join(
            SAVE_DIR,
            f"{idx}.pt"
        )

        torch.save(reps, rep_path)

        record = {

            "id": idx,

            "audio_path": wav_path,

            "language": row["language"],

            "labels": row["labels"],

            "label_counts": row["label_counts"],

            "rep_path": rep_path
        }

        pd.DataFrame([record]).to_csv(
            csv_out,
            mode="a",
            header=write_header,
            index=False
        )

        write_header = False

        # occasionally print progress

        if len(rows) % 20 == 0:
            print(f"Processed {len(rows)} new samples")

    except Exception as e:

        print(f"Error at {wav_path}: {e}")

        continue



print("Saved:", csv_out)

In [ ]:
import gc
import torch

def judge_qwen2_text(
    system_prompt,
    user_prompt,
):

    conversation = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    text = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=False
    )

    inputs = processor(
        text=text,
        return_tensors="pt",
        padding=True
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
            use_cache=False
        )

    pooled_reps = {}

    hidden_states = outputs.hidden_states

    for i, h in enumerate(hidden_states):

        pooled_reps[f"lm_{i}"] = (
            h.mean(dim=1)
             .detach()
             .cpu()
        )

    del outputs
    del hidden_states
    del inputs

    gc.collect()
    torch.cuda.empty_cache()

    return pooled_reps

In [ ]:
import os
import json
import torch
import pandas as pd
from tqdm import tqdm

BASE_DIR = "/content/drive/MyDrive/subprobe"

def resolve_path(p):
    p = str(p).replace("\\", "/").strip()

    if p.startswith("/content/"):
        return p

    return os.path.join(BASE_DIR, p)


CSV_PATH = os.path.join(BASE_DIR, "merged_subtle_emotion.csv")

SAVE_DIR = os.path.join(
    BASE_DIR,
    "task_ablation_qwen/text"
)
os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(CSV_PATH)

# random subset
df = df.sample(n=1000, random_state=42)

print("Total samples:", len(df))

# ============================================================
# RESUME
# ============================================================

processed = {
    os.path.splitext(f)[0]
    for f in os.listdir(SAVE_DIR)
    if f.endswith(".pt")
}

print(f"Already processed: {len(processed)}")

# ============================================================
# PROMPTS
# ============================================================

SYSTEM_PROMPT = "You are a helpful assistant."

BASE_PROMPT = (
    "What is the emotion in this speech?"
)

# ============================================================
# RUN
# ============================================================

rows = []
csv_out = os.path.join(
    SAVE_DIR,
    "task3_results.csv"
)

write_header = not os.path.exists(csv_out)
for idx, row in tqdm(df.iterrows(), total=len(df)):

    # ----------------------------------------
    # Skip processed samples
    # ----------------------------------------

    if str(idx) in processed:
        continue

    wav_path = resolve_path(row["audio_path"])

    if not os.path.exists(wav_path):
        print(f"Missing audio: {wav_path}")
        continue

    # ----------------------------------------
    # Parse labels
    # ----------------------------------------

    labels = json.loads(row["labels"])
    counts = json.loads(row["label_counts"])

    transcript = str(row["Utterance"]).strip()

    current_prompt = (
        BASE_PROMPT +
        "\n\nTranscript: " +
        transcript
    )


    # ----------------------------------------
    # Inference
    # ----------------------------------------

    try:

        reps = judge_qwen2_text(
            SYSTEM_PROMPT,
            current_prompt
        )

        # save representation

        rep_path = os.path.join(
            SAVE_DIR,
            f"{idx}.pt"
        )

        torch.save(reps, rep_path)

        record = {

            "id": idx,

            "audio_path": wav_path,

            "language": row["language"],

            "labels": row["labels"],

            "label_counts": row["label_counts"],

            "rep_path": rep_path
        }

        pd.DataFrame([record]).to_csv(
            csv_out,
            mode="a",
            header=write_header,
            index=False
        )

        write_header = False

        # occasionally print progress

        if len(rows) % 20 == 0:
            print(f"Processed {len(rows)} new samples")

    except Exception as e:

        print(f"Error at {wav_path}: {e}")

        continue



print("Saved:", csv_out)

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# CONFIG
# ============================================================

LAYER = "lm_15"

SPEECH_DIR = "/content/drive/MyDrive/subprobe/task_ablation_qwen/speech"

TEXT_DIR = "/content/drive/MyDrive/subprobe/task_ablation_qwen/text"

MULTI_DIR = "/content/drive/MyDrive/subprobe/task_ablation_qwen/multimodal"

# ============================================================
# LOAD EMBEDDING
# ============================================================

def load_embedding(path):

    rep = torch.load(path, map_location="cpu")

    if LAYER not in rep:
        return None

    x = rep[LAYER]

    if x.dim() == 3:
        x = x.mean(dim=1).squeeze(0)

    elif x.dim() == 2:
        x = x.squeeze(0)

    x = x.float()

    x = x / (x.norm() + 1e-8)

    return x.numpy()

# ============================================================
# FIND COMMON FILES
# ============================================================

speech_files = {
    f[:-3]
    for f in os.listdir(SPEECH_DIR)
    if f.endswith(".pt")
}

text_files = {
    f[:-3]
    for f in os.listdir(TEXT_DIR)
    if f.endswith(".pt")
}

multi_files = {
    f[:-3]
    for f in os.listdir(MULTI_DIR)
    if f.endswith(".pt")
}

common = sorted(
    speech_files &
    text_files &
    multi_files,
    key=int
)

print("Common samples:", len(common))

# ============================================================
# COMPUTE SIMILARITIES
# ============================================================

rows = []

for idx in tqdm(common):

    speech = load_embedding(
        os.path.join(SPEECH_DIR, idx + ".pt")
    )

    text = load_embedding(
        os.path.join(TEXT_DIR, idx + ".pt")
    )

    multi = load_embedding(
        os.path.join(MULTI_DIR, idx + ".pt")
    )

    if speech is None or text is None or multi is None:
        continue

    speech_text = cosine_similarity(
        speech.reshape(1,-1),
        text.reshape(1,-1)
    )[0,0]

    speech_multi = cosine_similarity(
        speech.reshape(1,-1),
        multi.reshape(1,-1)
    )[0,0]

    text_multi = cosine_similarity(
        text.reshape(1,-1),
        multi.reshape(1,-1)
    )[0,0]

    rows.append({

        "id": idx,

        "speech_text": speech_text,

        "speech_multimodal": speech_multi,

        "text_multimodal": text_multi

    })

# ============================================================
# RESULTS
# ============================================================

similarity_df = pd.DataFrame(rows)


print()

print(similarity_df.describe())

In [ ]:
rep = torch.load("/content/drive/MyDrive/subprobe/task_ablation_qwen/multimodal/100.pt")

print(rep["lm_2"].dtype)

In [ ]:
print(similarity_df[
    [
        "speech_text",
        "speech_multimodal",
        "text_multimodal"
    ]
].mean())

In [ ]:
speech = torch.load("/content/drive/MyDrive/subprobe/task_ablation_qwen/speech/100.pt")
multi  = torch.load("/content/drive/MyDrive/subprobe/task_ablation_qwen/multimodal/100.pt")

for k in ["lm_2"]:
    print(torch.equal(speech[k], multi[k]))
    print(torch.max(torch.abs(speech[k] - multi[k])))

speech = speech["lm_2"]
multi = multi["lm_2"]

print(speech.shape)
print(multi.shape)
diff = (speech - multi).abs().mean(dim=-1)

print(diff.shape)

In [ ]:
import torch

def inspect_pt(path, verbose=False):

    data = torch.load(path, map_location="cpu")

    print(f"\n📦 File: {path}")

    # If wrapped dict (recommended format)
    if isinstance(data, dict) and "reps" in data:
        reps = data["reps"]
        print("Contains metadata:")
        for k in data:
            if k != "reps":
                print(f"  {k}: {data[k]}")
    else:
        reps = data

    print("\n🔑 Keys:")
    for k in reps.keys():
        print(f"  - {k}")

    print("\n📊 Tensor Info:")
    for k, v in reps.items():

        if not isinstance(v, torch.Tensor):
            print(f"{k}: ❌ Not a tensor ({type(v)})")
            continue

        print(f"{k}: shape={tuple(v.shape)}, dtype={v.dtype}")

        if verbose:
            print(f"   mean={v.mean().item():.4f}, std={v.std().item():.4f}")

            # print small tensors fully
            if v.numel() < 20:
                print(f"   values={v}")

    print("\n✅ Done.\n")

In [ ]:
inspect_pt("/content/drive/MyDrive/subprobe/task_ablation_qwen/multimodal/5182.pt")

In [ ]:
# =========================================================
# WEIGHTED SUBTLE CENTROIDS
# =========================================================
#
# Goal:
#
# Build annotator-weighted subtle emotion
# centroids.
#
# Example:
#
# {"joy":2, "pride":2, "approval":1}
#
# contributes:
#
# joy      -> weight 2/5
# pride    -> weight 2/5
# approval -> weight 1/5
#
# =========================================================

import ast
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

# =========================================================
# CONFIG
# =========================================================

TARGET_LAYER = "lm_2"

MODALITIES = {

    "Speech": {
        "label_csv":
        "/content/drive/MyDrive/subprobe/task_ablation_qwen/speech/task3_results.csv",
        "rep_dir":
        "/content/drive/MyDrive/subprobe/task_ablation_qwen/speech"
    },

    "Text": {
        "label_csv":
        "/content/drive/MyDrive/subprobe/task_ablation_qwen/text/task3_results.csv",
        "rep_dir":
        "/content/drive/MyDrive/subprobe/task_ablation_qwen/text"
    },

    "Multimodal": {
        "label_csv":
        "/content/drive/MyDrive/subprobe/task_ablation_qwen/multimodal/task3_results.csv",
        "rep_dir":
        "/content/drive/MyDrive/subprobe/task_ablation_qwen/multimodal"
    }
}

# =========================================================
# PARSERS
# =========================================================

def parse_labels(x):

    try:
        return ast.literal_eval(x)

    except:
        return []

def parse_label_counts(x):

    try:
        return ast.literal_eval(x)

    except:
        return {}

# =========================================================
# EMBEDDING LOADER
# =========================================================

def process_rep(x):

    if not torch.is_tensor(x):
        x = torch.tensor(x)

    x = x.float()

    # -----------------------------------------------------
    # pooling
    # -----------------------------------------------------

    if x.dim() == 3:
        x = x.mean(dim=1).squeeze(0)

    elif x.dim() == 2:
        x = x.squeeze(0)

    # -----------------------------------------------------
    # normalize
    # -----------------------------------------------------

    x = x / (x.norm() + 1e-8)

    return x.numpy()

def load_embedding(path, layer):

    try:

        rep = torch.load(
            path,
            map_location="cpu"
        )

        if layer not in rep:
            return None

        return process_rep(
            rep[layer]
        )

    except:
        return None

# =========================================================
# BUILD WEIGHTED CENTROIDS
# =========================================================

modality_centroids = {}

for modality, cfg in MODALITIES.items():

    print("="*60)
    print(modality)
    print("="*60)

    df = pd.read_csv(cfg["label_csv"])

    df["rep_path"] = (
        df["rep_path"]
        .astype(str)
        .str.strip()
    )

    print("Samples:", len(df))

    # =====================================================
    # PARSE LABELS
    # =====================================================

    df["subtle_labels"] = (
        df["labels"]
        .apply(parse_labels)
    )

    df["label_count_dict"] = (
        df["label_counts"]
        .apply(parse_label_counts)
    )

    # =====================================================
    # COLLECT ALL SUBTLE LABELS
    # =====================================================

    all_subtle = set()

    for d in df["label_count_dict"]:

        for k in d.keys():
            all_subtle.add(k)

    print(
        "Unique subtle labels:",
        len(all_subtle)
    )

    # =====================================================
    # BUILD CENTROIDS
    # =====================================================

    centroids = {}

    for subtle in sorted(all_subtle):

        weighted_embs = []

        weights = []

        # -------------------------------------------------
        # iterate samples
        # -------------------------------------------------

        for _, row in tqdm(
            df.iterrows(),
            total=len(df),
            desc=subtle
        ):

            count_dict = row[
                "label_count_dict"
            ]

            # ---------------------------------------------
            # skip if subtle absent
            # ---------------------------------------------

            if subtle not in count_dict:
                continue

            # ---------------------------------------------
            # weight
            # ---------------------------------------------

            total_votes = sum(
                count_dict.values()
            )

            if total_votes == 0:
                continue

            weight = (
                count_dict[subtle]
                /
                total_votes
            )

            # ---------------------------------------------
            # embedding
            # ---------------------------------------------

            emb = load_embedding(
                row["rep_path"],
                TARGET_LAYER
            )

            if emb is None:
                continue

            weighted_embs.append(
                emb * weight
            )

            weights.append(weight)

        # -------------------------------------------------
        # finalize centroid
        # -------------------------------------------------

        if len(weighted_embs) == 0:
            continue

        centroid = (
            np.sum(
                weighted_embs,
                axis=0
            )
            /
            (
                np.sum(weights)
                + 1e-8
            )
        )

        # normalize
        centroid = (
            centroid
            /
            (
                np.linalg.norm(centroid)
                + 1e-8
            )
        )

        centroids[subtle] = centroid

        print(
            f"{subtle}: "
            f"{len(weights)} samples"
        )

    # =====================================================
    # SAVE MODEL CENTROIDS
    # =====================================================

    modality_centroids[modality] = centroids

    print(f"Built {len(centroids)} centroids")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

geometry_matrices = {}

for modality, centroids in modality_centroids.items():

    emotions = sorted(centroids.keys())

    mat = pd.DataFrame(
        index=emotions,
        columns=emotions,
        dtype=float
    )

    for e1 in emotions:
        for e2 in emotions:

            sim = cosine_similarity(
                centroids[e1].reshape(1, -1),
                centroids[e2].reshape(1, -1)
            )[0, 0]

            mat.loc[e1, e2] = sim

    geometry_matrices[modality] = mat

In [ ]:
for modality, mat in geometry_matrices.items():

    x = mat.values

    vals = x[np.triu_indices_from(x, k=1)]

    print(
        modality,
        vals.mean(),
        vals.std()
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

modalities = [
    "Speech",
    "Text",
    "Multimodal"
]

# --------------------------------------------------
# Shared color scale
# --------------------------------------------------

vmin = min(
    geometry_matrices[m].values.min()
    for m in modalities
)

vmax = max(
    geometry_matrices[m].values.max()
    for m in modalities
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18,6)
)

for ax, modality in zip(axes, modalities):

    mat = geometry_matrices[modality]

    im = ax.imshow(
        mat.values,
        vmin=vmin,
        vmax=vmax,
        aspect="auto"
    )

    ax.set_title(modality)

    ax.set_xticks(range(len(mat.columns)))
    ax.set_xticklabels(
        mat.columns,
        rotation=90,
        fontsize=7
    )

    ax.set_yticks(range(len(mat.index)))
    ax.set_yticklabels(
        mat.index,
        fontsize=7
    )

# --------------------------------------------------
# One shared colorbar
# --------------------------------------------------

fig.colorbar(
    im,
    ax=axes.ravel().tolist(),
    shrink=0.75,
    label="Cosine similarity"
)

plt.tight_layout()
plt.show()

In [ ]:
for modality, mat in geometry_matrices.items():

    x = mat.values

    # remove diagonal
    mask = ~np.eye(len(x), dtype=bool)

    vals = x[mask]

    print(modality)
    print("Mean:", vals.mean())
    print("Std :", vals.std())
    print("Min :", vals.min())
    print("Max :", vals.max())
    print()

In [ ]:
speech_multi = (
    geometry_matrices["Multimodal"]
    - geometry_matrices["Speech"]
)

text_multi = (
    geometry_matrices["Multimodal"]
    - geometry_matrices["Text"]
)

speech_text = (
    geometry_matrices["Speech"]
    - geometry_matrices["Text"]
)

In [ ]:
v = max(
    np.abs(speech_multi.values).max(),
    np.abs(text_multi.values).max(),
    np.abs(speech_text.values).max()
)

fig, axes = plt.subplots(1,3, figsize=(18,6))

diffs = {
    "Multi - Speech": speech_multi,
    "Multi - Text": text_multi,
    "Speech - Text": speech_text
}

for ax, (title, mat) in zip(axes, diffs.items()):

    im = ax.imshow(
        mat.values,
        cmap="bwr",
        vmin=-v,
        vmax=v
    )

    ax.set_title(title)

    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(
    im,
    ax=axes.ravel().tolist(),
    shrink=0.8,
    label="Δ cosine similarity"
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy.stats import pearsonr

def flatten_upper(mat):

    x = mat.values

    idx = np.triu_indices_from(x, k=1)

    return x[idx]

pairs = [
    ("Speech", "Text"),
    ("Speech", "Multimodal"),
    ("Text", "Multimodal")
]

for a, b in pairs:

    x = flatten_upper(geometry_matrices[a])
    y = flatten_upper(geometry_matrices[b])

    r, _ = pearsonr(x, y)

    print(a, b, r)

In [ ]:
diff = (
    geometry_matrices["Multimodal"]
    - geometry_matrices["Speech"]
).abs()

diff.stack().sort_values(ascending=False).head(20)

In [ ]:
import random
from sklearn.metrics.pairwise import cosine_similarity

paths = list(df["rep_path"])

pairs = random.sample(
    [(i, j) for i in range(len(paths)) for j in range(i+1, len(paths))],
    100
)

sims = []

for i, j in pairs:

    x = load_embedding(paths[i], TARGET_LAYER)
    y = load_embedding(paths[j], TARGET_LAYER)

    sims.append(
        cosine_similarity(
            x.reshape(1,-1),
            y.reshape(1,-1)
        )[0,0]
    )

print(np.mean(sims), np.std(sims))

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

geometry_matrices = {}

for modality, centroids in modality_centroids.items():

    emotions = sorted(centroids.keys())

    # stack centroids
    X = np.stack([
        centroids[e]
        for e in emotions
    ])

    # cosine similarity matrix
    sim = cosine_similarity(X)

    geometry_matrices[modality] = sim

    # ---------------------------------------------------
    # Average pairwise centroid similarity
    # (ignore diagonal)
    # ---------------------------------------------------

    vals = sim[np.triu_indices_from(sim, k=1)]

    print(
        modality,
        "mean =", vals.mean(),
        "std =", vals.std()
    )

In [ ]:
from scipy.stats import pearsonr

speech = geometry_matrices["Speech"]
multi = geometry_matrices["Multimodal"]
text = geometry_matrices["Text"]

idx = np.triu_indices_from(speech, k=1)

r_sm = pearsonr(speech[idx], multi[idx])[0]
r_st = pearsonr(speech[idx], text[idx])[0]
r_tm = pearsonr(text[idx], multi[idx])[0]

print("Speech vs Multimodal:", r_sm)
print("Speech vs Text:", r_st)
print("Text vs Multimodal:", r_tm)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')